# Block 4 — KG rescoring

Trusted ticks only (`is_marked` and not HiTL). Write-ins via `assume()`. Observed tubes from Block 3 digits — **never** copy expected into empty crops.

Do **not** upload clinic PHI.


## 0. Clone the live tree


In [ ]:
"""Colab/kernel bootstrap for live src/med_doc.

Opening a GitHub notebook does not clone the repo. Run this as the first cell.
Prints BOOTSTRAP_V3 when import med_doc succeeds.
"""

from __future__ import annotations

import os
import site
import subprocess
import sys
from pathlib import Path

BOOTSTRAP_VERSION = "BOOTSTRAP_V3"
REPO = "https://github.com/RwaRwa599/epq3.git"
BRANCH = "block1"


def _run(cmd: list[str]) -> None:
    print("$", " ".join(str(c) for c in cmd))
    subprocess.check_call(cmd)


def _token() -> str | None:
    tok = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata

        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


def _clone_url() -> str:
    tok = _token()
    if tok:
        return f"https://{tok}@github.com/RwaRwa599/epq3.git"
    return REPO


def _find_root() -> Path | None:
    here = Path.cwd().resolve()
    for cand in (
        here,
        here.parent,
        Path("/content/epq3"),
        Path("/content") / "epq3",
    ):
        if (cand / "src" / "med_doc" / "__init__.py").is_file():
            return cand
    return None


def _write_pth(src: Path) -> None:
    line = str(src.resolve()) + "\n"
    dirs = []
    try:
        dirs.extend(site.getsitepackages())
    except Exception:
        pass
    try:
        dirs.append(site.getusersitepackages())
    except Exception:
        pass
    sp = Path(sys.prefix) / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    dirs.append(str(sp))
    for d in dirs:
        if not d:
            continue
        target = Path(d)
        try:
            target.mkdir(parents=True, exist_ok=True)
            (target / "epq3_src.pth").write_text(line, encoding="utf-8")
            print("wrote", target / "epq3_src.pth")
        except Exception as exc:
            print("pth skip", target, exc)


def _ipython_cd(path: Path) -> None:
    try:
        ip = get_ipython()  # type: ignore[name-defined]
    except Exception:
        ip = None
    if ip is None:
        os.chdir(path)
        return
    ip.run_line_magic("cd", str(path))


def _ipython_pip(root: Path) -> None:
    try:
        ip = get_ipython()  # type: ignore[name-defined]
    except Exception:
        ip = None
    if ip is not None:
        ip.run_line_magic("pip", "install -q matplotlib opencv-python-headless pydantic Pillow numpy")
        ip.run_line_magic("pip", f"install -q -e {root}")
        return
    _run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "opencv-python-headless", "pydantic", "Pillow", "numpy"])
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)])


def put_src_on_path(root: Path | None = None) -> Path:
    root = root or _find_root()
    if root is None:
        raise ModuleNotFoundError(
            "med_doc not found. Run the first notebook cell (BOOTSTRAP_V3 clone). "
            "Private repo: Colab secret GITHUB_TOKEN. Then Runtime → Run all."
        )
    src = (root / "src").resolve()
    os.chdir(root)
    if str(src) not in sys.path:
        sys.path.insert(0, str(src))
    os.environ["PYTHONPATH"] = str(src) + os.pathsep + os.environ.get("PYTHONPATH", "")
    return root


def bootstrap() -> Path:
    print(BOOTSTRAP_VERSION)
    dest = Path("/content/epq3") if Path("/content").is_dir() else (Path.cwd().resolve() / "epq3")
    root = _find_root()
    if root is None:
        url = _clone_url()
        if dest.exists() and not (dest / "src" / "med_doc" / "__init__.py").is_file():
            import shutil

            shutil.rmtree(dest, ignore_errors=True)
        if not (dest / ".git").is_dir():
            _run(["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", url, str(dest)])
        else:
            _run(["git", "-C", str(dest), "fetch", "origin", BRANCH])
            _run(["git", "-C", str(dest), "checkout", BRANCH])
            _run(["git", "-C", str(dest), "pull", "--ff-only", "origin", BRANCH])
        root = dest
    _ipython_cd(root)
    os.chdir(root)
    src = root / "src"
    if str(src.resolve()) not in sys.path:
        sys.path.insert(0, str(src.resolve()))
    _write_pth(src)
    try:
        _ipython_pip(root)
    except Exception as exc:
        print("pip note:", exc)
    # Drop a copy next to cwd as last resort (some Colab kernels ignore .pth until restart)
    try:
        import med_doc  # noqa: F401
    except ModuleNotFoundError:
        sys.path.insert(0, str(src.resolve()))
        import importlib

        importlib.invalidate_caches()
        import med_doc  # noqa: F401
    import med_doc

    print("cwd:", os.getcwd())
    print("med_doc:", med_doc.__file__)
    if "src" not in Path(med_doc.__file__).parts:
        print("warning: unexpected med_doc location")
    return root


root = bootstrap()


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet

In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

from med_doc.htr.batch import process_from_block1
from med_doc.kg import KnowledgeGraph
from med_doc.normalization.batch import normalize_batch
from med_doc.rescoring import process_from_block3
from med_doc.review import ReviewPatch, process_from_block4

kg = KnowledgeGraph.load()

def ensure_block1():
    z = OUT / "block1.zip"
    if z.exists():
        return z
    return Path(normalize_batch([demo_sheet()], output_dir=OUT / "b1", output_zip=z)["output_zip"])

def ensure_block3():
    z = OUT / "block3.zip"
    if z.exists():
        return z
    b1 = ensure_block1()
    return Path(process_from_block1(b1, output_dir=OUT / "b3", output_zip=z, kg=kg, backend="lexicon", mode="both")["output_zip"])

def ensure_block4():
    z = OUT / "block4.zip"
    if z.exists():
        return z
    b3 = ensure_block3()
    return Path(process_from_block3(b3, output_dir=OUT / "b4", output_zip=z, kg=kg)["output_zip"])

## 1. Rescore Block 3 drafts


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

b4 = process_from_block3(ensure_block3(), output_dir=OUT / "b4", output_zip=OUT / "block4.zip", kg=kg)
doc = b4["manifest"]["documents"][0]
print("ticked", doc["ticked_test_ids"])
print("expected", doc["expected_tubes"])
print("observed", doc["observed_tubes"])
print("hitl", doc["hitl_fields"])
print("discrepancies", doc["discrepancies"])
pred = json.loads((OUT / "b4" / "docs" / doc["doc_id"] / "prediction.json").read_text())
hyp = json.loads((OUT / "b4" / "docs" / doc["doc_id"] / "hypotheses.json").read_text())
print("hypotheses tube source", hyp["verbal"].get("tube_edta", {}).get("source"))
assert hyp["verbal"].get("tube_edta", {}).get("source") != "prior_expected"
print("prediction tube canonical", pred["handwriting_fields"].get("tube_edta", {}).get("canonical_value"))
download(OUT / "block4.zip")